In [5]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 2.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import re
from gensim.models import FastText
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load the larger spaCy model for better accuracy
nlp = spacy.load('en_core_web_lg')

# Load your preprocessed text file
file_path = 'preprocessed_text.txt'  # Change this to your file path
with open(file_path, 'r') as file:
    text_data = file.read()

# Step 1: Preprocessing the Text
def preprocess_text(text):
    sentences = text.lower().split('.')
    tokenized_sentences = []
    for sentence in sentences:
        tokens = re.findall(r'\b\w+\b', sentence)
        tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS and len(token) > 2]
        if tokens:
            tokenized_sentences.append(tokens)
    return tokenized_sentences

# Apply preprocessing
tokenized_sentences = preprocess_text(text_data)

# Step 2: Train FastText Embeddings
fasttext_model = FastText(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=1)

# Step 3: Dependency Parsing
doc = nlp(text_data)

# Extract SVO relations
def extract_svo(doc):
    svos = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = [w for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w for w in token.rights if w.dep_ in ("dobj", "attr", "prep")]
            if subj and obj:
                svos.append((subj[0], token, obj[0]))
    return svos

svo_relations = extract_svo(doc)

# Step 4: Filter SVO Relationships to Keep Only Important Concepts
important_concepts = ['risk', 'management', 'uncertainty', 'mitigation', 'control', 'project', 'threat']
filtered_svos = [(subj, verb, obj) for subj, verb, obj in svo_relations
                 if subj.text in important_concepts or obj.text in important_concepts]

# Step 5: Combine SVO with FastText Embeddings
def get_combined_embeddings(svo_relations, model):
    combined_embeddings = []
    for subj, verb, obj in svo_relations:
        subj_vec = model.wv[subj.text] if subj.text in model.wv else None
        verb_vec = model.wv[verb.text] if verb.text in model.wv else None
        obj_vec = model.wv[obj.text] if obj.text in model.wv else None

        if subj_vec is not None and verb_vec is not None and obj_vec is not None:
            combined_embeddings.append({
                'subject': subj.text,
                'verb': verb.text,
                'object': obj.text,
                'subject_embedding': subj_vec,
                'verb_embedding': verb_vec,
                'object_embedding': obj_vec
            })

    return combined_embeddings

# Get combined embeddings for filtered SVOs
combined_results = get_combined_embeddings(filtered_svos, fasttext_model)

# Display the first few combined results
for result in combined_results[:5]:  # Show top 5 results
    print(f"Subject: {result['subject']}, Verb: {result['verb']}, Object: {result['object']}")
    print(f"Subject Embedding: {result['subject_embedding']}")
    print(f"Verb Embedding: {result['verb_embedding']}")
    print(f"Object Embedding: {result['object_embedding']}\n")

Subject: aspect, Verb: cover, Object: management
Subject Embedding: [-0.4608546   0.24979275 -0.24591434  0.09509107  0.22024027  0.3469093
  0.3172706   0.07882335  0.24942571 -0.25607476 -0.29595146 -0.00233164
 -0.12051966  0.39007792  0.29956678 -0.06255561  0.18755113  0.00452043
 -0.10629505 -0.11039818 -0.15833262  0.310799    0.02278516 -0.08248978
 -0.01713093 -0.24603547  0.21419267  0.06132181  0.29628295 -0.36214924
 -0.05299043  0.23874007 -0.46615008 -0.17440768 -0.09274807 -0.27636212
  0.22156924  0.30241573 -0.3660976   0.15186654  0.05984636 -0.34276262
  0.1749433  -0.00176058 -0.11306963  0.0516101   0.11354611  0.06893306
  0.06713372  0.23332646  0.0295799   0.01704818 -0.23402028 -0.1413596
  0.23057835  0.3391734  -0.27242145 -0.49318117 -0.0739047   0.09731378
  0.02351625 -0.2921985  -0.26378238 -0.13248424  0.07294592  0.25254458
  0.01021186  0.0725496   0.00199507  0.13881184 -0.109529   -0.00057881
  0.24214287 -0.56725144  0.13228522 -0.3105869  -0.271092

In [3]:
svo_relations

[(identi, ed, in),
 (edition, targeted, at),
 (who, participate, in),
 (supervisor, describes, input),
 (it, provides, information),
 (signi, process, tool),
 (who, perform, process),
 (chapter, includes, purpose),
 (that, recognized, practice),
 (b, provide, standard),
 (aspect, cover, management),
 (it, applied, to),
 (standard, cover, risk),
 (it, aligned, with),
 (hierarchy, organized, in),
 (analysis, perform, response),
 (each, described, in),
 (that, address, topic),
 (which, used, in),
 (principle, stated, at),
 (tool, evolving, principle),
 (that, contains, series),
 (practice, tailored, to),
 (that, comply, with),
 (a, presented, in),
 (arrangement, found, on),
 (principle, used, check),
 (management, includes, process),
 (edition, state, objective),
 (it, occurs, effect),
 (consideration, described, in),
 (it, applied, to),
 (it, becomes, part),
 (many, planning, project),
 (process, assume, degree),
 (it, build, upon),
 (budgeting, performed, at),
 (management, provides, ba

#Updated Code with Filtering

In [1]:
import spacy
import re
from gensim.models import FastText
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load the larger spaCy model for better accuracy
nlp = spacy.load('en_core_web_lg')

# Load your preprocessed text file
file_path = 'preprocessed_text.txt'
with open(file_path, 'r') as file:
    text_data = file.read()

# Step 1: Preprocessing the Text
def preprocess_text(text):
    sentences = text.lower().split('.')
    tokenized_sentences = []
    for sentence in sentences:
        tokens = re.findall(r'\b\w+\b', sentence)
        tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS and len(token) > 2]
        if tokens:
            tokenized_sentences.append(tokens)
    return tokenized_sentences

# Apply preprocessing
tokenized_sentences = preprocess_text(text_data)

# Step 2: Train FastText Embeddings
fasttext_model = FastText(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=1)

# Step 3: Dependency Parsing
doc = nlp(text_data)

# Extract SVO relations
def extract_svo(doc):
    svos = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = [w for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w for w in token.rights if w.dep_ in ("dobj", "attr", "prep")]
            if subj and obj:
                svos.append((subj[0], token, obj[0]))
    return svos

svo_relations = extract_svo(doc)

# **Add Filtering to Remove Generic Terms**
generic_words = {'it', 'this', 'they', 'provides', 'chapter', 'includes','t', 'according', 'upon', 'at',
                 'by', 'a', 'describes', 'many', 'one', 'that', 'to', 'so', 't', 'in', 'into', 'for', 'on', 'with'}
uninformative_verbs = {'used', 'provides', 'includes', 'aligned', 'requires', 'involved', 'affect', 'have',
                       'include', 'represent', 'known', 'pose'}
vague_terms = {'which', 'these', 'that', 'such'}

# Filter out generic terms, uninformative verbs, and vague subjects/objects
filtered_svos = [(subj, verb, obj) for subj, verb, obj in svo_relations
                 if subj.text not in generic_words and obj.text not in generic_words
                 and verb.text not in uninformative_verbs and subj.text not in vague_terms
                 and obj.text not in vague_terms]

# Step 4: Filter SVO Relationships to Keep Only Important Concepts
important_concepts = ['risk', 'management', 'uncertainty', 'mitigation', 'control', 'project', 'threat', 'exposure', 'outcome']
filtered_svos = [(subj, verb, obj) for subj, verb, obj in filtered_svos
                 if subj.text in important_concepts or obj.text in important_concepts]

# Remove duplicates
filtered_svos = list(set(filtered_svos))

# Display the first few SVO relationships as text (without embeddings)
for subj, verb, obj in filtered_svos[:10]:  # Show top 10 results
    print(f"Subject: {subj.text}, Verb: {verb.text}, Object: {obj.text}")


Subject: aspect, Verb: cover, Object: management
Subject: diagram, Verb: identify, Object: risk
Subject: customer, Verb: share, Object: risk
Subject: project, Verb: identifying, Object: risk
Subject: risk, Verb: result, Object: from
Subject: process, Verb: permit, Object: risk
Subject: standard, Verb: cover, Object: risk
Subject: review, Verb: predicted, Object: exposure
Subject: risk, Verb: assess, Object: importance
Subject: management, Verb: bsipd, Object: forum


In [3]:
# Display the first 70 SVO relationships as text (without embeddings)
for subj, verb, obj in filtered_svos[:70]:  # Show top 70 results
    print(f"Subject: {subj.text}, Verb: {verb.text}, Object: {obj.text}")

Subject: aspect, Verb: cover, Object: management
Subject: diagram, Verb: identify, Object: risk
Subject: customer, Verb: share, Object: risk
Subject: project, Verb: identifying, Object: risk
Subject: risk, Verb: result, Object: from
Subject: process, Verb: permit, Object: risk
Subject: standard, Verb: cover, Object: risk
Subject: review, Verb: predicted, Object: exposure
Subject: risk, Verb: assess, Object: importance
Subject: management, Verb: bsipd, Object: forum
Subject: risk, Verb: represents, Object: effect
Subject: tool, Verb: identify, Object: risk
Subject: risk, Verb: occurs, Object: information
Subject: risk, Verb: contribute, Object: most
Subject: focus, Verb: excludes, Object: risk
Subject: area, Verb: exhibit, Object: risk
Subject: management, Verb: carried, Object: part
Subject: risk, Verb: identify, Object: risk
Subject: standard, Verb: identify, Object: risk
Subject: template, Verb: presented, Object: risk
Subject: management, Verb: viewed, Object: process
Subject: item,

In [ ]:
"""#EXTRACTION CODE
import fitz  # PyMuPDF
import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from pdfminer.high_level import extract_text as pdfminer_extract_text
from spellchecker import SpellChecker

# Download required NLTK resources (only needs to be done once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize tools
lemmatizer = WordNetLemmatizer()
spell = SpellChecker()


# Check if the PDF is encrypted and try to extract text using PyMuPDF
def check_pdf_permissions_and_extract_text_with_pymupdf(pdf_document, start_page=13):
    doc = fitz.open(pdf_document)

    # Check if the document is encrypted
    if doc.is_encrypted:
        print("The document is encrypted.")
        password = input("Enter the PDF password (if available): ")
        if doc.authenticate(password):
            print("Document decrypted successfully.")
        else:
            print("Incorrect password. Text extraction might not work.")
            return None
    else:
        print("The document is not encrypted.")

    extracted_text = ""
    # Loop through the pages starting from the specified page (page 13)
    for page_num in range(start_page - 1, doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")

        if text.strip():
            extracted_text += text + "\n"
        else:
            print(f"Warning: No text found on page {page_num + 1}. Trying 'blocks'")
            text_blocks = page.get_text("blocks")
            for block in text_blocks:
                if block[-1] != 1:  # If not an image
                    extracted_text += block[4] + "\n"

    doc.close()
    return extracted_text


# Tokenization: Break down text into tokens (words, phrases, sentences)
def tokenize_text(text):
    sentences = sent_tokenize(text)  # Tokenize into sentences
    words = word_tokenize(text)  # Tokenize into words
    return sentences, words


# Data Cleaning: Remove unnecessary characters and correct misspellings
def clean_text(text):
    # Remove special characters and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Correct spelling
    words = text.split()
    corrected_words = [spell.correction(word) if word in spell else word for word in words]

    cleaned_text = ' '.join(corrected_words)
    return cleaned_text


# Lemmatization: Reduce words to their base forms
def lemmatize_words(words):
    lemmatized_words = [lemmatizer.lemmatize(word.lower()) for word in words]
    return lemmatized_words


# Full preprocessing pipeline
def preprocess_text(text):
    # Tokenization
    sentences, words = tokenize_text(text)

    # Data Cleaning
    cleaned_text = clean_text(text)

    # Lemmatization
    cleaned_words = word_tokenize(cleaned_text)
    lemmatized_words = lemmatize_words(cleaned_words)

    return {
        'sentences': sentences,
        'words': lemmatized_words,
        'cleaned_text': ' '.join(lemmatized_words)
    }


# Function to save preprocessed text to a file
def save_preprocessed_text(output_filename, preprocessed_data):
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(preprocessed_data['cleaned_text'])
    print(f"Preprocessed text saved to {output_filename}")


# Main function to extract, clean, and preprocess text from the PDF
def extract_and_preprocess_pdf(pdf_document, start_page=13):
    # Try to extract text using PyMuPDF
    print(f"Attempting to extract text using PyMuPDF from page {start_page} onwards...")
    extracted_text = check_pdf_permissions_and_extract_text_with_pymupdf(pdf_document, start_page)

    if extracted_text:
        # Preprocess the extracted text
        preprocessed_data = preprocess_text(extracted_text)
        save_preprocessed_text("preprocessed_text.txt", preprocessed_data)
    else:
        print("PyMuPDF failed to extract text. Trying pdfminer.six.")
        # Try to extract text using pdfminer.six
        extracted_text = extract_text_with_pdfminer(pdf_document)
        if extracted_text:
            preprocessed_data = preprocess_text(extracted_text)
            save_preprocessed_text("preprocessed_text.txt", preprocessed_data)
        else:
            print("Both PyMuPDF and pdfminer.six failed to extract text.")


# Example usage
pdf_document = "practice-standard-project-risk-management.pdf"
extract_and_preprocess_pdf(pdf_document, start_page=13)
"""

In [ ]:
"""#NER CODE:
import spacy
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load spaCy's English model for NER, POS, and dependency parsing
nlp = spacy.load("en_core_web_sm")

# Perform Named Entity Recognition (NER) and Part-of-Speech Tagging (POS)
def perform_ner_and_pos(text):
    doc = nlp(text)
    ner_results = []
    pos_results = []

    # Extract NER
    for ent in doc.ents:
        ner_results.append((ent.text, ent.label_))

    # Extract POS tags
    for token in doc:
        pos_results.append((token.text, token.pos_))

    return ner_results, pos_results

# Perform Dependency Parsing with improved results
def perform_dependency_parsing(text):
    doc = nlp(text)
    dependencies = []
    for token in doc:
        dependencies.append((token.text, token.dep_, token.head.text, [child.text for child in token.children]))
    return dependencies

# Function to save lists to text files
def save_list_to_file(output_filename, data_list):
    with open(output_filename, "w", encoding="utf-8") as f:
        for item in data_list:
            f.write(f"{item}\n")
    print(f"Data saved to {output_filename}")

# Analyze preprocessed text for NER, POS, and dependencies
def analyze_preprocessed_text(preprocessed_text):
    # Perform Named Entity Recognition (NER) and Part-of-Speech Tagging (POS)
    ner_results, pos_results = perform_ner_and_pos(preprocessed_text)

    # Save results to text files
    save_list_to_file("ner_results.txt", [f"{entity}: {label}" for entity, label in ner_results])
    save_list_to_file("pos_results.txt", [f"{word}: {pos}" for word, pos in pos_results])

    # Perform and save Dependency Parsing results
    dependencies = perform_dependency_parsing(preprocessed_text)
    save_list_to_file("dependencies.txt", [f"{word} -> {dep} -> {head} -> {children}" for word, dep, head, children in dependencies])

# Function to calculate TF-IDF and save results
def calculate_tfidf(text, threshold=0.01):
    # Create a TfidfVectorizer, ignoring English stop words
    vectorizer = TfidfVectorizer(stop_words='english')

    # Fit and transform the text data into TF-IDF matrix
    tfidf_matrix = vectorizer.fit_transform([text])

    # Create a DataFrame for better readability
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

    # Get the first row of TF-IDF scores and filter by the threshold
    tfidf_scores = tfidf_df.iloc[0]
    filtered_words = tfidf_scores[tfidf_scores >= threshold]

    # Save TF-IDF results to a text file
    with open("tfidf_results.txt", "w", encoding="utf-8") as f:
        f.write("Term\tTF-IDF Score\n")
        f.write("---------------------\n")
        for word, score in filtered_words.items():
            f.write(f"{word}\t{score}\n")
    print(f"TF-IDF results with threshold {threshold} saved to tfidf_results.txt")

    # Save TF-IDF results to a CSV file
    tfidf_df.to_csv("tfidf_results.csv", index=False)
    print("TF-IDF results saved to tfidf_results.csv")

# Main function to handle NER, POS, TF-IDF, and dependency parsing
def main():
    # Load preprocessed text from file
    with open('preprocessed_text.txt', 'r', encoding='utf-8') as f:
        preprocessed_text = f.read()

    # Step 1: Analyze the preprocessed text for NER and POS
    analyze_preprocessed_text(preprocessed_text)

    # Step 2: Calculate and save TF-IDF results with a threshold of 0.01
    calculate_tfidf(preprocessed_text, threshold=0.09)

# Run the main function
if __name__ == "__main__":
    main()
"""

In [ ]:
"""#EXTRACTING CONCEPS:
import re
from collections import defaultdict


def extract_concepts_from_dependencies(dependencies):
    concept_patterns = [
        r'\b(\w+(?:\s+\w+){0,2})\s+(?:is|are|was|were|can be)\s+(?:defined|described|characterized|understood|viewed|considered|regarded)\s+as\s+(.*?)(?:\.|\n)',
        r'\b(\w+(?:\s+\w+){0,2})\s+refers\s+to\s+(.*?)(?:\.|\n)',
        r'\b(\w+(?:\s+\w+){0,2})\s+(?:means|implies|denotes|signifies|represents)\s+(.*?)(?:\.|\n)',
        r'(\w+(?:\s+\w+){0,2})\s+(?:includes|encompasses|comprises|consists of|involves)\s+(.*?)(?:\.|\n)',
        r'(\w+(?:\s+\w+){0,2})\s+(?:is|are)\s+(?:a|an|the)\s+(.*?)(?:\.|\n)',
        r'(?:The term|The concept of)\s+["\']?(\w+(?:\s+\w+){0,2})["\']?\s+(?:refers to|describes|denotes)\s+(.*?)(?:\.|\n)'
    ]

    concepts = []
    for pattern in concept_patterns:
        matches = re.findall(pattern, dependencies, flags=re.IGNORECASE | re.DOTALL)
        for match in matches:
            concept = match[0].strip().capitalize()
            definition = match[1].strip()
            if definition and len(definition) > 10:  # Ensure there's a substantial definition
                concepts.append((concept, definition))

    return concepts


def main():
    # Load dependencies from file
    with open('dependencies.txt', 'r', encoding='utf-8') as f:
        dependencies = f.read()

    concepts = extract_concepts_from_dependencies(dependencies)

    print(f"Extracted {len(concepts)} unique concepts:\n")
    for concept, definition in concepts:
        print(f"Concept: {concept}")
        print(f"  Definition: {definition}")
        print()


if __name__ == "__main__":
    main()"""